# Mayavi visualization of MuMax3 output
Itxaso Muñoz-Aldalur

`conda env list`

`conda activate my_mayavi_env_py37`

> **Dependencies**: only `numpy`, `mayavi`, `vtk` — no ubermag needed.
> Plain-text spin files (`x y z mx my mz`) are produced by the mumax3 analysis notebook (§1 OVF→txt).

## 0. Configuration

In [ ]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt

# Template
plt.style.use('/Users/itxaso/MASTER-codi/2-TFM/MUMAX3/mplstyle/science.mplstyle')

# ── EDIT ──────────────────────────────────────────────────────────────
BASE_DIR = "/Users/itxaso/MASTER-codi/2-TFM/mumax3/"

# Simulation folder (must contain the exported m*.txt files)
SIM_DIR = os.path.join(BASE_DIR, "setA-Cortes-Ortuno/setA-KLscan-uniaxial.out/")

# Output directory for rendered images
IMAGES_DIR = os.path.join(SIM_DIR, "mayavi_images")
os.makedirs(IMAGES_DIR, exist_ok=True)

# Cylinder geometry (nm): adjust to match nanotube
CYL_RADIUS  = 3.6013 / 2   # half outer-diameter, nm
CYL_HEIGHT  = 10.606 * 2   # full length, nm

# Glyph style
GLYPH_MODE  = 'cube'  # 'sphere', 'cone', 'arrow', 'cube_source'

# Scalar component used for colouring: 'mx', 'my', 'mz'
COLOUR_COMP = 'mz'

# Scan grid dimensions (must match what you used in mumax3)
N_D = 11
N_H = 11
# ── END EDIT ───────────────────────────────────────────────────────────────

# Locate exported plain-text spin files (previously produced m*.txt)
txt_files = sorted(glob.glob(os.path.join(SIM_DIR, "m*.txt")))
print(f"SIM_DIR    : {SIM_DIR}")
print(f"IMAGES_DIR : {IMAGES_DIR}")
print(f"Spin files : {len(txt_files)}  (expected {N_D * N_H})")
if txt_files:
    sample = np.loadtxt(txt_files[0])
    print(f"Columns    : {sample.shape[1]}  (x y z mx my mz)")
    print(f"Spins/file : {sample.shape[0]}")


SIM_DIR    : /Users/itxaso/MASTER-codi/2-TFM/mumax3/setA-Cortes-Ortuno/setA-KLscan-uniaxial-min.out/
IMAGES_DIR : /Users/itxaso/MASTER-codi/2-TFM/mumax3/setA-Cortes-Ortuno/setA-KLscan-uniaxial-min.out/mayavi_images
Spin files : 0  (expected 121)


## 1. Mayavi

In [2]:
%gui qt
import vtk
import mayavi
from mayavi import mlab
import numpy as np

mlab.options.backend = 'envisage'

# Build the custom LUT (plasma + dark-blue for the low end)
x, y = np.mgrid[-10:10:200j, -10:10:200j]
z_dummy = 100 * np.sin(x * y) / (x * y + 1e-9)
mlab.figure(bgcolor=(0.82, 0.82, 0.82))
_surf = mlab.surf(z_dummy, colormap='plasma')
lut = _surf.module_manager.scalar_lut_manager.lut.table.to_array().copy()
dark_blue = [51, 102, 153, 255]
for i in range(30):
    lut[i] = dark_blue
_surf.module_manager.scalar_lut_manager.lut.table = lut
mlab.close()   # close dummy figure; lut is kept in memory
print('LUT ready.')


/Applications/anaconda3/envs/my_mayavi_env_py37/lib/python3.7/site-packages/pyface/ui/qt/workbench/split_tab_widget.py:125: UserWarning: Attempting to restore to a non-empty widget.
  warnings.warn("Attempting to restore to a non-empty widget.")


LUT ready.


## 2. Helper functions

All rendering functions accept a plain numpy array with columns `[x, y, z, mx, my, mz]`.

In [3]:
def _load(txt_path):
    """Return (x,y,z,mx,my,mz) arrays from a plain-text spin file."""
    pts = np.loadtxt(txt_path)
    return pts[:,0], pts[:,1], pts[:,2], pts[:,3], pts[:,4], pts[:,5]


def _scalar(mx, my, mz, comp='mz'):
    """Pick the scalar array used for colouring."""
    return {'mx': mx, 'my': my, 'mz': mz}[comp]


def _scene_setup(fig):
    """Apply lighting and isometric camera to a figure."""
    try:
        engine = mlab.get_engine()
    except NameError:
        from mayavi.api import Engine
        engine = Engine()
        engine.start()
    if len(engine.scenes) == 0:
        engine.new_scene()
    scene = fig.scene
    scene.isometric_view()
    scene.light_manager.lights[1].intensity = 1.0
    scene.light_manager.lights[2].intensity = 1.0
    scene.render()
    return scene


def _draw_cylinder(radius=CYL_RADIUS, height=CYL_HEIGHT):
    """Draw a semi-transparent cylindrical surface guide."""
    theta = np.linspace(0, 2*np.pi, 100)
    z_ax  = np.linspace(-height/2, height/2, 50)
    Theta, Z = np.meshgrid(theta, z_ax)
    X = radius * np.cos(Theta)
    Y = radius * np.sin(Theta)
    cyl = mlab.mesh(X, Y, Z, color=(0.85, 0.9, 0.82), opacity=0.5)
    cyl.actor.mapper.scalar_visibility = False
    return cyl


def _apply_lut_and_colourbar(res, title='mz'):
    """Attach the custom LUT and configure the colour bar."""
    try:
        res.module_manager.scalar_lut_manager.lut.table = lut
    except NameError:
        print('Warning: lut not defined — using default colormap.')
    res.glyph.glyph_source.glyph_position = 'center'
    res.glyph.color_mode = 'color_by_scalar'
    res.module_manager.scalar_lut_manager.scalar_bar.title = title
    res.module_manager.scalar_lut_manager.label_text_property.font_size  = 8
    res.module_manager.scalar_lut_manager.label_text_property.italic     = False
    res.module_manager.scalar_lut_manager.title_text_property.font_size  = 7


## 3. 3D isometric view (cylinder)

Renders each spin file as a 3-D nanotube with the cylinder guide, isometric camera.

##### $m_z$

In [5]:
%gui qt
mlab.options.backend = 'envisage'

def render_3d_isometric(txt_path, save=True, scalar='mphi'):
    points = np.loadtxt(txt_path)
    x, y, z   = points[:, 0], points[:, 1], points[:, 2]
    mx, my, mz = points[:, 3], points[:, 4], points[:, 5]

    # --- compute scalar to colour by ---
    if scalar == 'mphi':
        cx, cy = (x.min()+x.max())/2, (y.min()+y.max())/2
        phi_pos = np.arctan2(y - cy, x - cx)
        s = -mx * np.sin(phi_pos) + my * np.cos(phi_pos)
        label = 'mphi'
    else:
        s = mz
        label = 'mz'

    fig = mlab.figure(size=(1000, 1000), bgcolor=(0.82, 0.82, 0.82))
    res = mlab.quiver3d(x, y, z, mx, my, mz,
                        scalars=s,
                        line_width=3, mode='cube',
                        colormap='coolwarm',      # diverging colormap
                        scale_factor=1.5,
                        vmin=-1, vmax=1)          # fixed range for all lengths

    res.glyph.glyph_source.glyph_position = 'center'
    res.glyph.color_mode  = 'color_by_scalar'
    res.glyph.glyph.orient       = False
    res.glyph.glyph.scale_factor = 1.0

    # lock the LUT range so it does not auto-rescale per file
    lut_mgr = res.module_manager.scalar_lut_manager
    lut_mgr.use_default_range = False             # key line
    lut_mgr.data_range = (-1.0, 1.0)             # same for every length

    try:
        engine = mlab.get_engine()
    except NameError:
        from mayavi.api import Engine
        engine = Engine(); engine.start()
    if len(engine.scenes) == 0:
        engine.new_scene()

    scene = engine.scenes[0]
    scene.scene.isometric_view()
    scene = fig.scene
    scene.light_manager.lights[1].intensity = 1.0
    scene.light_manager.lights[2].intensity = 1.0
    scene.render()

    if save:
        base = os.path.splitext(os.path.basename(txt_path))[0]
        out  = os.path.join(IMAGES_DIR, f"{label}_{base}_3d_iso.png")
        mlab.savefig(out, size=(1000, 1000))
        print(f"Saved: {out}")

    #mlab.close(fig)


# --- run ---
for f in txt_files:
    render_3d_isometric(f)

Saved: /Users/itxaso/MASTER-codi/2-TFM/mumax3/setA-Cortes-Ortuno/setA-KLHscan-cubic.out/mayavi_images/mphi_m_H0_Lz112nm_mayavi_3d_iso.png
Saved: /Users/itxaso/MASTER-codi/2-TFM/mumax3/setA-Cortes-Ortuno/setA-KLHscan-cubic.out/mayavi_images/mphi_m_H0_Lz128nm_mayavi_3d_iso.png
Saved: /Users/itxaso/MASTER-codi/2-TFM/mumax3/setA-Cortes-Ortuno/setA-KLHscan-cubic.out/mayavi_images/mphi_m_H0_Lz16nm_mayavi_3d_iso.png
Saved: /Users/itxaso/MASTER-codi/2-TFM/mumax3/setA-Cortes-Ortuno/setA-KLHscan-cubic.out/mayavi_images/mphi_m_H0_Lz32nm_mayavi_3d_iso.png
Saved: /Users/itxaso/MASTER-codi/2-TFM/mumax3/setA-Cortes-Ortuno/setA-KLHscan-cubic.out/mayavi_images/mphi_m_H0_Lz48nm_mayavi_3d_iso.png
Saved: /Users/itxaso/MASTER-codi/2-TFM/mumax3/setA-Cortes-Ortuno/setA-KLHscan-cubic.out/mayavi_images/mphi_m_H0_Lz64nm_mayavi_3d_iso.png
Saved: /Users/itxaso/MASTER-codi/2-TFM/mumax3/setA-Cortes-Ortuno/setA-KLHscan-cubic.out/mayavi_images/mphi_m_H0_Lz80nm_mayavi_3d_iso.png
Saved: /Users/itxaso/MASTER-codi/2-TFM/

##### $m_\phi$

## 4. Top-down view (−z)

Same 3-D data viewed from above the nanotube axis.

In [ ]:
def render_3d_zview(txt_path, save=True, scalar='mphi'):
    points = np.loadtxt(txt_path)
    x, y, z   = points[:, 0], points[:, 1], points[:, 2]
    mx, my, mz = points[:, 3], points[:, 4], points[:, 5]

    # --- compute scalar to colour by ---
    if scalar == 'mphi':
        cx, cy = (x.min()+x.max())/2, (y.min()+y.max())/2
        phi_pos = np.arctan2(y - cy, x - cx)
        s = -mx * np.sin(phi_pos) + my * np.cos(phi_pos)
        label = 'mphi'
    else:
        s = mz
        label = 'mz'

    fig = mlab.figure(size=(1000, 1000), bgcolor=(0.82, 0.82, 0.82))
    res = mlab.quiver3d(x, y, z, mx, my, mz,
                        scalars=s,
                        line_width=3, mode=GLYPH_MODE,
                        colormap='coolwarm',      # diverging colormap
                        scale_factor=1.5,
                        vmin=-1, vmax=1)          # fixed range for all lengths

    res.glyph.glyph_source.glyph_position = 'center'
    res.glyph.color_mode  = 'color_by_scalar'
    res.glyph.glyph.orient       = False
    res.glyph.glyph.scale_factor = 1.0

    lut_mgr = res.module_manager.scalar_lut_manager
    lut_mgr.use_default_range = False             # key line
    lut_mgr.data_range = (-1.0, 1.0)             # same for every length

    try:
        engine = mlab.get_engine()
    except NameError:
        from mayavi.api import Engine
        engine = Engine(); engine.start()
    if len(engine.scenes) == 0:
        engine.new_scene()

    scene = fig.scene
    scene.z_minus_view()
    scene.light_manager.lights[1].intensity = 1.0
    scene.light_manager.lights[2].intensity = 1.0
    scene.render()

    if save:
        base = os.path.splitext(os.path.basename(txt_path))[0]
        out  = os.path.join(IMAGES_DIR, f"{label}_{base}_zview.png")
        mlab.savefig(out, size=(2000, 2000))
        print(f"Saved: {out}")

    #mlab.close(fig)


# --- run ---
for f in txt_files:
    render_3d_zview(f)


Saved: /Users/itxaso/MASTER-codi/2-TFM/mumax3/setA-Cortes-Ortuno/setA-Lscan.out/mayavi_images/m_Lz112nm_mayavi_zview_mphi.png
Saved: /Users/itxaso/MASTER-codi/2-TFM/mumax3/setA-Cortes-Ortuno/setA-Lscan.out/mayavi_images/m_Lz128nm_mayavi_zview_mphi.png
Saved: /Users/itxaso/MASTER-codi/2-TFM/mumax3/setA-Cortes-Ortuno/setA-Lscan.out/mayavi_images/m_Lz16nm_mayavi_zview_mphi.png
Saved: /Users/itxaso/MASTER-codi/2-TFM/mumax3/setA-Cortes-Ortuno/setA-Lscan.out/mayavi_images/m_Lz32nm_mayavi_zview_mphi.png
Saved: /Users/itxaso/MASTER-codi/2-TFM/mumax3/setA-Cortes-Ortuno/setA-Lscan.out/mayavi_images/m_Lz48nm_mayavi_zview_mphi.png
Saved: /Users/itxaso/MASTER-codi/2-TFM/mumax3/setA-Cortes-Ortuno/setA-Lscan.out/mayavi_images/m_Lz64nm_mayavi_zview_mphi.png
Saved: /Users/itxaso/MASTER-codi/2-TFM/mumax3/setA-Cortes-Ortuno/setA-Lscan.out/mayavi_images/m_Lz80nm_mayavi_zview_mphi.png
Saved: /Users/itxaso/MASTER-codi/2-TFM/mumax3/setA-Cortes-Ortuno/setA-Lscan.out/mayavi_images/m_Lz96nm_mayavi_zview_mphi.pn

## 5. Unfolded plane (work in progress)

Maps cylindrical coordinates to a flat plane: `xp = r·atan2(y,x)`, `yp = z`.

In [8]:
def render_plane_angle(txt_path, save=True):
    x0, y0, z0, sx, sy, sz = _load(txt_path)

    r    = np.sqrt(x0**2 + y0**2)
    phi  = np.arctan2(y0, x0)
    xp   = r * phi
    yp   = z0
    zp   = np.zeros_like(xp)

    # rotate spin vectors to planar frame
    sx2 = -sx * np.sin(phi) + sy * np.cos(phi)   # tangential
    sy2 =  sz                                      # axial → new y
    sz2 =  sx * np.cos(phi) + sy * np.sin(phi)    # radial

    s = _scalar(sx2, sy2, sz2, COLOUR_COMP)

    fig = mlab.figure(size=(2000, 1000), bgcolor=(0.82, 0.82, 0.82))
    res = mlab.quiver3d(xp, yp, zp, sx2, sy2, sz2,
                        scalars=s, line_width=3,
                        mode=GLYPH_MODE, colormap='plasma', scale_factor=0.8)
    _apply_lut_and_colourbar(res, title=COLOUR_COMP)

    scene = fig.scene
    scene.z_plus_view()
    scene.light_manager.lights[1].intensity = 1.0
    scene.light_manager.lights[2].intensity = 1.0
    scene.render()

    if save:
        base = os.path.splitext(os.path.basename(txt_path))[0]
        out  = os.path.join(IMAGES_DIR, f"Plane_{base}_angle.png")
        mlab.savefig(out, size=(5000, 3000))
        print(f"Saved: {out}")

    #mlab.close(fig)


# --- run ---
for f in txt_files:
    render_plane_angle(f)


ERROR|2026-05-30 16:45:24,154|Exception occurred in traits notification handler for object: <envisage.plugins.python_shell.view.python_shell_view.PythonShellView object at 0x7fabf8d68fb0>, trait: stdout_text, old value: <undefined>, new value: Saved: /Users/itxaso/MASTER-codi/2-TFM/mumax3/setA-Cortes-Ortuno/setA-Hrad-2.out/Images-mayavi/Plane_m000000_mayavi_angle.png
Traceback (most recent call last):
  File "/Applications/anaconda3/envs/my_mayavi_env_py37/lib/python3.7/site-packages/traits/trait_notifiers.py", line 524, in _dispatch_change_event
    self.dispatch(handler, *args)
  File "/Applications/anaconda3/envs/my_mayavi_env_py37/lib/python3.7/site-packages/traits/trait_notifiers.py", line 619, in dispatch
    handler(*args)
  File "/Applications/anaconda3/envs/my_mayavi_env_py37/lib/python3.7/site-packages/envisage/plugins/python_shell/view/python_shell_view.py", line 238, in _on_write_stdout
    self.shell.control.write(text)
AttributeError: 'NoneType' object has no attribute 'w

## 6. Unfolded plane: zig-zag topology

Same unfolding but coloured by `my` (axial component) and uses arrow glyphs.

In [13]:
def render_plane_angle(txt_path, save=True):
    points = np.loadtxt(txt_path)
    x0, y0, z0 = points[:, 0], points[:, 1], points[:, 2]
    sx, sy, sz = points[:, 3], points[:, 4], points[:, 5]

    r   = np.sqrt(x0**2 + y0**2)
    phi = np.arctan2(y0, x0)

    xp  = r * phi           # arc-length along circumference
    yp  = z0                # axial direction
    zp  = r - r.mean()      # ← wall thickness centred on 0 (thin shell → ~0, thick wall → spread)

    # spin vector rotation to planar frame
    sx2 = -sx * np.sin(phi) + sy * np.cos(phi)   # tangential
    sy2 =  sz                                      # axial → new y
    sz2 =  sx * np.cos(phi) + sy * np.sin(phi)    # radial → new z

    fig = mlab.figure(size=(1000, 1000), bgcolor=(0.82, 0.82, 0.82))
    res = mlab.quiver3d(xp, yp, zp, sx2, sy2, sz2,
                        scalars=sx2,
                        line_width=3, mode='sphere',
                        colormap='viridis', scale_factor=0.8)
    res.glyph.glyph_source.glyph_position = 'center'
    res.glyph.color_mode = 'color_by_scalar'

    try:
        engine = mlab.get_engine()
    except NameError:
        from mayavi.api import Engine
        engine = Engine()
        engine.start()
    if len(engine.scenes) == 0:
        engine.new_scene()

    scene = engine.scenes[0]
    # look straight down onto the xp-yp plane, ignoring wall thickness
    scene.scene.z_plus_view()
    scene = fig.scene
    scene.light_manager.lights[1].intensity = 1.0
    scene.light_manager.lights[2].intensity = 1.0
    scene.render()

    if save:
        base = os.path.splitext(os.path.basename(txt_path))[0]
        out  = os.path.join(IMAGES_DIR, f"Plane_{base}_angle.png")
        mlab.savefig(out, size=(1000, 1000))
        print(f"Saved: {out}")

    #mlab.close(fig)


# --- run ---
for f in txt_files:
    render_plane_angle(f)

ERROR|2026-05-30 17:11:07,261|Exception occurred in traits notification handler for object: <envisage.plugins.python_shell.view.python_shell_view.PythonShellView object at 0x7f919071a410>, trait: stdout_text, old value: <undefined>, new value: Saved: /Users/itxaso/MASTER-codi/2-TFM/mumax3/setA-Cortes-Ortuno/setA-Hrad-2.out/mayavi_images/Plane_m000001_mayavi_angle.png
Traceback (most recent call last):
  File "/Applications/anaconda3/envs/my_mayavi_env_py37/lib/python3.7/site-packages/traits/trait_notifiers.py", line 524, in _dispatch_change_event
    self.dispatch(handler, *args)
  File "/Applications/anaconda3/envs/my_mayavi_env_py37/lib/python3.7/site-packages/traits/trait_notifiers.py", line 619, in dispatch
    handler(*args)
  File "/Applications/anaconda3/envs/my_mayavi_env_py37/lib/python3.7/site-packages/envisage/plugins/python_shell/view/python_shell_view.py", line 238, in _on_write_stdout
    self.shell.control.write(text)
AttributeError: 'NoneType' object has no attribute 'w

qt.qpa.fonts: Populating font family aliases took 724 ms. Replace uses of missing font family "Monaco" with one that exists to avoid this cost. 


## 7. Single-file interactive preview

Run this cell to inspect one specific snapshot interactively (no auto-save).

In [ ]:
# Change the index to pick a different snapshot
PREVIEW_IDX = 0

if txt_files:
    f = txt_files[PREVIEW_IDX]
    print(f"Previewing: {f}")
    x, y, z, mx, my, mz = _load(f)
    s = _scalar(mx, my, mz, COLOUR_COMP)

    fig = mlab.figure(size=(800, 800), bgcolor=(0.82, 0.82, 0.82))
    res = mlab.quiver3d(x, y, z, mx, my, mz,
                        scalars=s, line_width=3,
                        mode=GLYPH_MODE, colormap='plasma', scale_factor=0.8)
    _apply_lut_and_colourbar(res, title=COLOUR_COMP)
    _draw_cylinder()
    _scene_setup(fig)
    mlab.show()   # opens interactive window; close it to continue
else:
    print('No txt_files found — check SIM_DIR and run §1 (OVF→txt) first.')
